# C_DeepSUM Master Script

In [1]:
#!/usr/bin/env python
# coding: utf-8

import os
import re
import sys
import glob
import json
import time
import importlib.util
from pathlib import Path
from datetime import timedelta
from collections import defaultdict

import tensorflow as tf
tf.compat.v1.disable_eager_execution()

%reload_ext autoreload
%autoreload 2

2026-06-19 11:27:21.401361: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-19 11:27:21.428145: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-19 11:27:22.038671: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
nickname = "yellow"

config_path_manual = None

n_epochs_manual = 4

C_ROOT = Path("/misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM")
C_GLOBAL_FILES_DIR = C_ROOT / "C_global_files"
C_RUNS_DIR = C_ROOT / "C_runs"

CD_PATH = C_GLOBAL_FILES_DIR / "CD_1.py"
UDN_PATH = C_GLOBAL_FILES_DIR / "UDN_1.py"
SR_PATH = C_GLOBAL_FILES_DIR / "SR_2.py"

os.chdir(C_GLOBAL_FILES_DIR)

if str(C_GLOBAL_FILES_DIR) not in sys.path:
    sys.path.insert(0, str(C_GLOBAL_FILES_DIR))

print("Working directory:", os.getcwd())
print("C_GLOBAL_FILES_DIR:", C_GLOBAL_FILES_DIR)
print("CD_PATH:", CD_PATH)
print("UDN_PATH:", UDN_PATH)
print("SR_PATH:", SR_PATH)

Working directory: /misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_global_files
C_GLOBAL_FILES_DIR: /misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_global_files
CD_PATH: /misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_global_files/CD_1.py
UDN_PATH: /misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_global_files/UDN_1.py
SR_PATH: /misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_global_files/SR_2.py


In [3]:
def import_module_from_path(module_name: str, module_path: Path):
    module_path = Path(module_path).resolve()

    if not module_path.exists():
        raise FileNotFoundError(f"No existe el módulo: {module_path}")

    spec = importlib.util.spec_from_file_location(module_name, str(module_path))
    module = importlib.util.module_from_spec(spec)

    if spec.loader is None:
        raise ImportError(f"No se pudo cargar el módulo: {module_path}")

    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module


def get_latest_config_path(nickname: str, root_runs_dir: Path) -> str:
    config_dir = Path(root_runs_dir) / nickname / "config_files"
    pattern = str(config_dir / f"config_{nickname}_v*.json")
    candidates = glob.glob(pattern)

    if not candidates:
        raise FileNotFoundError(
            f"No se encontraron configs para nickname='{nickname}' en: {config_dir}"
        )

    def extract_version(path: str) -> int:
        name = Path(path).name
        match = re.search(r"_v(\d+)\.json$", name)
        if not match:
            return -1
        return int(match.group(1))

    valid = [p for p in candidates if extract_version(p) >= 0]

    if not valid:
        raise ValueError(
            f"Se encontraron archivos en {config_dir}, pero ninguno cumple formato config_<nickname>_vX.json"
        )

    return max(valid, key=extract_version)


def pick_config_value(data: dict, key: str, default=None, required: bool = True):
    if key in data:
        return data[key]

    for section in ["hyperparameters", "others", "paths", "dataset", "training"]:
        if section in data and isinstance(data[section], dict) and key in data[section]:
            return data[section][key]

    if required and default is None:
        raise KeyError(f"No se encontró la clave requerida '{key}' en el config.")

    return default


def build_train_config(data: dict, config_path: str) -> defaultdict:
    config = defaultdict()

    config["lr"] = pick_config_value(data, "lr")
    config["batch_size"] = pick_config_value(data, "batch_size")
    config["skip_step"] = pick_config_value(data, "skip_step")

    config["nickname"] = pick_config_value(data, "nickname")
    config["hiresl_h"] = pick_config_value(data, "hiresl_h")
    config["hiresl_w"] = pick_config_value(data, "hiresl_w")

    config["channels"] = pick_config_value(data, "channels")
    config["T_in"] = pick_config_value(data, "T_in")
    config["R"] = pick_config_value(data, "R")
    config["full"] = pick_config_value(data, "full")
    config["patch_size_HR"] = pick_config_value(data, "patch_size_HR")
    config["patch_size_LR"] = pick_config_value(data, "patch_size_LR")
    config["border"] = pick_config_value(data, "border")
    config["spectral_band"] = pick_config_value(data, "spectral_band")

    config["dataset_path"] = pick_config_value(data, "dataset_path")
    config["n_chunks"] = pick_config_value(data, "n_chunks")

    config["RegNet_pretrain_dir"] = pick_config_value(data, "RegNet_pretrain_dir", default="", required=False)
    config["SISRNet_pretrain_dir"] = pick_config_value(data, "SISRNet_pretrain_dir", default="", required=False)

    config["mu"] = pick_config_value(data, "mu")
    config["sigma"] = pick_config_value(data, "sigma")
    config["sigma_rescaled"] = pick_config_value(data, "sigma_rescaled")

    tensorboard_dir = pick_config_value(data, "tensorboard_dir", default=None, required=False)

    if tensorboard_dir is None:
        tensorboard_dir = (
            f"{config['nickname'].upper()}_"
            f"{config['spectral_band']}_"
            f"lr_{config['lr']}_"
            f"bsize_{config['batch_size']}"
        )

    config["tensorboard_dir"] = tensorboard_dir
    config["config_path"] = config_path
    return config

In [4]:
if config_path_manual:
    config_path = str(config_path_manual)
    mode = "manual"

    config_name = Path(config_path).name
    match = re.match(r"config_(.+)_v\d+\.json$", config_name)
    if match:
        nickname = match.group(1)
else:
    config_path = get_latest_config_path(nickname, C_RUNS_DIR)
    mode = "auto_latest"

print(f"Modo de selección: {mode}")
print(f"Nickname: {nickname}")
print(f"Config path: {config_path}")

with open(config_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Config cargado correctamente.")
print("Top-level keys:", list(data.keys()))

others = data.get("others", {})
print("dataset_path:", others.get("dataset_path"))
print("testset_path:", others.get("testset_path"))
print("mat_path:", others.get("mat_path"))
print("n_chunks:", others.get("n_chunks"))
print("scale:", others.get("scale"))
print("spectral_band:", others.get("spectral_band"))

Modo de selección: auto_latest
Nickname: yellow
Config path: /misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_runs/yellow/config_files/config_yellow_v1.json
Config cargado correctamente.
Top-level keys: ['hyperparameters', 'others']
dataset_path: /misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_runs/yellow/dataset_yellow/
testset_path: /misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_runs/yellow/testset_yellow/
mat_path: /misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_global_files/cmats/real_data_DEEPSUM_12subj_9reps_scale3_20251020.mat
n_chunks: 5
scale: 3
spectral_band: NIR


In [5]:
CD_1 = import_module_from_path("CD_1", CD_PATH)

print("CD_1 importado desde:")
print(CD_1.__file__ if hasattr(CD_1, "__file__") else CD_PATH)

try:
    if hasattr(CD_1, "main"):
        used_config = CD_1.main(config_path=config_path)
    elif hasattr(CD_1, "run_create_dataset"):
        used_config = CD_1.run_create_dataset(config_path=config_path)
    else:
        raise AttributeError(
            "CD_1.py no tiene función main(config_path=...) ni run_create_dataset(config_path=...)."
        )

    print("CREATE_DATASET se ejecutó correctamente con el config indicado.")

except Exception as e:
    print("CREATE_DATASET falló.")
    print(f"Error: {e}")
    raise

CD_1 importado desde:
/misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_global_files/CD_1.py
Nuevo nickname: yellow
dataset path: /misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_runs/yellow/dataset_yellow
testset path: /misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_runs/yellow/testset_yellow
nickname: yellow
spectral band: NIR
mat_path: /misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_global_files/cmats/real_data_DEEPSUM_12subj_9reps_scale3_20251020.mat

slices: 1440
num_subj: 12
train_subj: 10
imgs_per_subject: 120
registration_ref: 1
shift_abs_threshold: 4.0
scale: 3

n_chunks: 5
patch_size: 96
num_patches_per_set: 20
smart_patching: True
Carpeta Dataset: /misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_runs/yellow/dataset_yellow
Carpeta Testset: /misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_runs/yellow/testset_yellow
LR: (1440, 9, 86, 86) HR: (1440, 258, 258) MLR: (1440, 9, 86, 86) MHR: (1440, 258, 258) NORM: (1440, 1)
Train: 1200 Val: 120 Test: 120


/misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_global_files/UDN_1.py:197: ComplexWarning: Casting complex values to real discards the imaginary part
  imageset_training_registered[j_index]=corrected_image


[TRAIN] removed img indices per set: {}
[TRAIN] removed sets (<9 imgs): []
[TRAIN] final LR sets: 1200
[TRAIN] final HR shape: (1200, 258, 258)
[VAL] removed img indices per set: {}
[VAL] removed sets (<9 imgs): []
[VAL] final LR sets: 120
[VAL] final HR shape: (120, 258, 258)


[==========                                                              ]  14%

[TEST] removed img indices per set: {}
[TEST] removed sets (<9 imgs): []
[TEST] final LR sets: 120
[TEST] final HR shape: (120, 258, 258)


[========================================================================] 100%
[========================================================================] 100%
[========================================================================] 100%
[========================================================================] 100%
[========================================================================] 100%


Found LR patch files: 5
Found HR patch files: 5

Config updated successfully.
Config path: /misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_runs/yellow/config_files/config_yellow_v1.json
Old values: {'mu': 7754.664, 'sigma': 2316.3408, 'sigma_rescaled': 596.982}
New values: {'mu': 350.2608379703318, 'sigma': 321.5150391243168, 'sigma_rescaled': 110.20486640185854}
Pixels used -> LR: 9953280000, HR: 1105920000
Create_Dataset se ejecutó con éxito
CREATE_DATASET se ejecutó correctamente con el config indicado.


In [5]:
UDN_1 = import_module_from_path("UDN_1", UDN_PATH)

print("UDN_1 importado desde:")
print(UDN_1.__file__ if hasattr(UDN_1, "__file__") else UDN_PATH)

if not hasattr(UDN_1, "SR_network"):
    raise AttributeError("UDN_1.py no contiene SR_network.")

SR_network = UDN_1.SR_network

# Recargamos el config porque create_dataset puede actualizar mu/sigma/sigma_rescaled.
with open(config_path, "r", encoding="utf-8") as f:
    data = json.load(f)

config = build_train_config(data, config_path)

print("CONFIG PATH =>", config["config_path"])
print("DATASET PATH =>", config["dataset_path"])
print("nickname =>", config["nickname"])
print("hiresl_h =>", config["hiresl_h"])
print("hiresl_w =>", config["hiresl_w"])
print("spectral_band =>", config["spectral_band"])
print("tensorboard_dir =>", config["tensorboard_dir"])

if n_epochs_manual is not None:
    n_epochs = n_epochs_manual
else:
    n_epochs = pick_config_value(data, "n_epochs", default=1, required=False)

print("n_epochs =>", n_epochs)

UDN_1 importado desde:
/misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_global_files/UDN_1.py
CONFIG PATH => /misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_runs/yellow/config_files/config_yellow_v1.json
DATASET PATH => /misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_runs/yellow/dataset_yellow/
nickname => yellow
hiresl_h => 258
hiresl_w => 258
spectral_band => NIR
tensorboard_dir => YELLOW_NIR_lr_5e-06_bsize_3
n_epochs => 4


In [6]:
import tensorflow.compat.v1 as tfv1

tfv1.reset_default_graph()

model = SR_network(config)
model.build()

print("Modelo construido correctamente.")
print("Dataset path desde model =>", model.dataset_path)
print("Nickname desde model =>", model.nickname)
print("Checkpoint dir esperado =>", os.path.abspath("checkpoints/" + config["tensorboard_dir"]))

I0000 00:00:1781890073.305006 2871104 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5934 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070, pci bus id: 0000:01:00.0, compute capability: 8.9


Modelo construido correctamente.
Dataset path desde model => /misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_runs/yellow/dataset_yellow/
Nickname desde model => yellow
Checkpoint dir esperado => /misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_global_files/checkpoints/YELLOW_NIR_lr_5e-06_bsize_3


In [ ]:
start_time = time.time()
print("Entrenamiento iniciado...")

try:
    step = model.train(n_epochs=n_epochs)

    end_time = time.time()
    elapsed_seconds = end_time - start_time
    elapsed_time = str(timedelta(seconds=elapsed_seconds))

    print("\n------------------------------------------")
    print("ENTRENAMIENTO COMPLETADO")
    print("------------------------------------------")
    print(f"Último step: {step}")
    print(f"Tiempo total: {elapsed_time} (hh:mm:ss)")
    print("------------------------------------------\n")

except Exception as e:
    end_time = time.time()
    elapsed_seconds = end_time - start_time
    elapsed_time = str(timedelta(seconds=elapsed_seconds))

    print("\n------------------------------------------")
    print("TRAIN FALLÓ")
    print("------------------------------------------")
    print(f"Error: {e}")
    print(f"Tiempo transcurrido: {elapsed_time} (hh:mm:ss)")
    print("------------------------------------------\n")
    raise

Entrenamiento iniciado...


I0000 00:00:1781890187.631102 2871104 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled


No checkpoint found. Starting training from scratch.
Retrieving portion of training...
(24000, 9, 96, 96, 1)


2026-06-19 11:33:05.145002: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91300


Training Loss for a mini batch at step 999: 0.23019187152385712
Score on test batch at epoch 0 (lower is better): 488.0804443359375 
cPSNR on test batch at epoch 0: 65.03446960449219 dB
SSIM on test batch at epoch 0: 0.4361451268196106 
Filter exact accuracy on test batch at epoch 0: 0.07500000298023224 
Filter ±1 px accuracy on test batch at epoch 0: 0.6822916865348816 
Filter shift L1 error on test batch at epoch 0: 1.256250023841858 
Took: 21.637609481811523 seconds


In [ ]:
try:
    if hasattr(model, "sess") and model.sess is not None:
        print("Cerrando sesión TensorFlow del entrenamiento...")
        model.sess.close()
except Exception as e:
    print("No se pudo cerrar la sesión:")
    print(e)

In [ ]:
SR_2 = import_module_from_path("SR_2", SR_PATH)

print("SR_2 importado desde:")
print(SR_2.__file__ if hasattr(SR_2, "__file__") else SR_PATH)

try:
    if hasattr(SR_2, "run_superresolve"):
        sr_result = SR_2.run_superresolve(config_path=Path(config_path))
    elif hasattr(SR_2, "main"):
        try:
            sr_result = SR_2.main(config_path=Path(config_path))
        except TypeError:
            sr_result = SR_2.main()
    else:
        raise AttributeError(
            "SR_2.py no tiene función run_superresolve(config_path=...) ni main(...)."
        )

    print("SUPERRESOLVE se ejecutó correctamente.")

except Exception as e:
    print("SUPERRESOLVE falló.")
    print(f"Error: {e}")
    raise

In [ ]:
print("MASTER COMPLETO FINALIZADO")